# Silver -- `stg_time_of_day`

One minute of the day, 0-1439, carrying the hour it falls in.

**Source:** `bronze_kastle_pg_time_of_day`  
**Business key:** `time_key`  
**Load pattern:** `full_refresh`

> GENERATED FILE -- DO NOT EDIT.
Produced by framework/generators/generate_notebooks.py from the project spec set. Edit the spec and regenerate; hand edits are overwritten and will fail the notebook-lint gate.


In [ ]:
# Parameters -- overridden per environment by the deployment pipeline.
# See 05-deployment.yaml `parameterisation`.
# Reads from lh_bronze, writes to lh_silver. Both must be
# attached to this notebook; lh_silver must be the DEFAULT so an
# unqualified write cannot land in the wrong item.
target_item = "lh_silver"
source_item = "lh_bronze"
environment = "dev"
dq_failure_action = "warn"

import sys
from datetime import datetime

from pyspark.sql import functions as F

from ttfabric.cleansing import RuleContext, get_rule
from ttfabric.quality import DQRunLog

load_id = f"load_{datetime.utcnow():%Y%m%d_%H%M%S}"

def resolve_table(name: str):
    """Resolve a spec table reference to a DataFrame.

    Deliberately UNQUALIFIED, so the read lands in the default lakehouse.

    Rules reference tables in their OWN layer -- enforce_referential_integrity
    against dim_products, recompute_total_from_lines against fct_order_items --
    and those peers live in the item this notebook writes to, not the one it
    reads its source from. Qualifying with source_item sent them to
    lh_bronze.dim_products, which does not and should not exist.

    The single cross-item read, this table's own bronze source, is qualified
    explicitly at the call site instead.
    """
    bare = name.split(".")[-1]
    return spark.read.table(bare)

ctx = RuleContext(
    spark=spark,
    load_id=load_id,
    environment=environment,
    table="stg_time_of_day",
    resolve_table=resolve_table,
    apply_masking=(environment in ("uat", "prod")),
)

dq = DQRunLog(spark, load_id=load_id, layer="silver", table_name="stg_time_of_day")
print(f"load_id={load_id}  environment={environment}  table=stg_time_of_day")


In [ ]:
# ---- Read bronze -------------------------------------------------
df = spark.read.table(f"{source_item}.bronze_kastle_pg_time_of_day")
rows_in = df.count()
dq.record_input(rows_in)
print(f"read {rows_in:,} rows from bronze_kastle_pg_time_of_day")


In [ ]:
# ---- Column mapping ----------------------------------------------
# Lifted verbatim from mappings/silver.yaml so the notebook is
# self-contained and auditable without opening the spec.
columns = [   {   'source': 'time_key',
        'target': 'time_key',
        'type': 'integer',
        'nullable': False,
        'key': 'business'},
    {'source': 'hour', 'target': 'hour_of_day', 'type': 'integer', 'nullable': False}]


In [ ]:
# ---- Cleansing rules ---------------------------------------------
# Each rule returns kept and rejected rows. Rejected rows accumulate
# into the quarantine frame so nothing is lost without a reason.
quarantine = None

def apply(result):
    """Collect rejects and carry the kept frame forward."""
    global quarantine, df
    if result.rejected is not None and not result.rejected.isEmpty():
        quarantine = (result.rejected if quarantine is None
                      else quarantine.unionByName(result.rejected,
                                                  allowMissingColumns=True))
    if result.corrected_count:
        dq.record_corrected(result.corrected_count)
    df = result.kept
    return result


In [ ]:
# cast_types
result = apply(get_rule("cast_types")(df, ctx, columns=columns))
dq.record_rule("cast_types", result)


In [ ]:
# add_record_hash
result = apply(get_rule("add_record_hash")(df, ctx, exclude=['_processed_at', '_load_id']))
dq.record_rule("add_record_hash", result)


In [ ]:
# ---- Write -------------------------------------------------------
final_columns = [c["target"] for c in columns]
out = df.select(*[c for c in final_columns if c in df.columns])

# Audit columns from 00-platform.yaml `audit_columns.silver`.
out = (out
    .withColumn("_processed_at", F.current_timestamp())
    .withColumn("_load_id", F.lit(load_id)))

rows_out = out.count()
out.write.mode("overwrite").option("overwriteSchema", "true") \
    .format("delta").saveAsTable("stg_time_of_day")
dq.record_output(rows_out)
print(f"wrote {rows_out:,} rows to stg_time_of_day")

if quarantine is not None:
    rejected_count = quarantine.count()
    # Overwrite, matching silver's own write mode. Silver is fully
    # rebuilt each run, so an appended quarantine would accumulate
    # rejects from previous builds and break the layer-level identity
    # count(bronze) == count(silver) + count(quarantine).
    (quarantine.write.mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta").saveAsTable("stg_time_of_day_quarantine"))
    dq.record_quarantined(rejected_count)
    print(f"quarantined {rejected_count:,} rows to stg_time_of_day_quarantine")
else:
    rejected_count = 0


In [ ]:
# ---- Reconciliation ----------------------------------------------
# SILVER-RECON-003: every input row is accounted for. A shortfall
# means a rule dropped rows without quarantining them, which is a
# framework bug rather than a data problem.
accounted = rows_out + rejected_count
if accounted != rows_in:
    raise AssertionError(
        f"row loss: {rows_in:,} in, {rows_out:,} out, "
        f"{rejected_count:,} quarantined, {rows_in - accounted:,} unaccounted"
    )
print(f"reconciled: {rows_in:,} = {rows_out:,} kept + {rejected_count:,} quarantined")

dq.flush()
